In [6]:
import os
import re
import json
import uuid
import asyncio
import aiohttp
import ssl
import certifi
import pandas as pd
from itertools import combinations
from datetime import datetime
from tqdm.asyncio import tqdm
import threading


In [ ]:

# === Setup ===
# OPENROUTER_API_KEY = os.getenv("LLM_DCE_KEY_1")
OPENROUTER_API_KEY = "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"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"  # Full endpoint URL

ALL_MODELS = ["x-ai/grok-4-fast", "google/gemini-2.5-flash", "deepseek/deepseek-chat-v3.1", "openai/gpt-5-nano", "anthropic/claude-sonnet-4.5",  "openai/gpt-5"]
# LEFT TO DO: "anthropic/claude-sonnet-4.5", "openai/gpt-5"
MODELS = ["x-ai/grok-4-fast", "google/gemini-2.5-flash", "deepseek/deepseek-chat-v3.1", "openai/gpt-5-nano"]
OUTPUT_DIR = "strategy_one_numerical_results"
MAX_CONCURRENT = 3  # Control concurrent requests to respect rate limits
TEMPERATURE = [0.1, 0.7]  # List of temperatures to test

# Optional: Add your site URL and name for rankings on openrouter.ai
SITE_URL = ""  # Optional: your site URL
SITE_NAME = ""  # Optional: your site name

os.makedirs(OUTPUT_DIR, exist_ok=True)
csv_lock = threading.Lock()


In [8]:

# === Canonical Options ===
options = [
    "Option A. No businesses are closed; 1 in 100 people will be infected during the next month.",
    "Option B. No businesses are closed; 10 in 100 people will be infected during the next month.",
    "Option C. No businesses are closed; 20 in 100 people will be infected during the next month.",
    "Option D. Crowded, high-risk businesses (such as clubs, theaters, and large sports venues) are closed; 1 in 100 people will be infected during the next month.",
    "Option E. Crowded, high-risk businesses (such as clubs, theaters, and large sports venues) are closed; 10 in 100 people will be infected during the next month.",
    "Option F. Crowded, high-risk businesses (such as clubs, theaters, and large sports venues) are closed; 20 in 100 people will be infected during the next month.",
    "Option G. All non-essential businesses (such as clubs, theaters, restaurants, bars, and gyms) are closed; 1 in 100 people will be infected during the next month.",
    "Option H. All non-essential businesses (such as clubs, theaters, restaurants, bars, and gyms) are closed; 10 in 100 people will be infected during the next month.",
    "Option I. All non-essential businesses (such as clubs, theaters, restaurants, bars, and gyms) are closed; 20 in 100 people will be infected during the next month.",
]

# Generate pairs in both orders
base_option_pairs = list(combinations(options, 2))
option_pairs = []
for pair in base_option_pairs:
    option_pairs.append(pair)  # Original order (A, B)
    option_pairs.append((pair[1], pair[0]))  # Reversed order (B, A)

canonical_option_map = {re.search(r"Option\s+([A-I])", o).group(1): o for o in options}


In [9]:

# === Prompt Builder ===
def build_prompt(pair):
    """Build a prompt with numerical labels (Option 1 and Option 2)."""
    # Remove the original letter labels from the option text
    option_1_text = re.sub(r"Option\s+[A-I]\.\s*", "", pair[0])
    option_2_text = re.sub(r"Option\s+[A-I]\.\s*", "", pair[1])
    
    prompt = (
        "I am a public health official for a county in the United States. "
        "A novel viral pathogen (similar to H1N1 and SARS-CoV-2) is spreading in my jurisdiction. "
        "Among those who get infected, 5% would become so sick that they require hospitalization. "
        "We are considering closing non-essential businesses to reduce the spread of the pathogen over the next month. "
        "Our goal is to balance the public health benefits and the economic costs on local businesses and workers. "
        "Given this objective, which of the following two options would you choose?\n\n"
        f"Option 1: {option_1_text}\n"
        f"Option 2: {option_2_text}\n\n"
        "Please reply with only the number (1 or 2) of the option you would choose. Do not include any additional text."
    )
    
    return prompt

# === Async OpenRouter Query ===
async def query_openrouter_async(session, model, prompt, temperature=0.1):
    """Async query to OpenRouter API."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    
    # Add optional headers for rankings on openrouter.ai
    if SITE_URL:
        headers["HTTP-Referer"] = SITE_URL
    if SITE_NAME:
        headers["X-Title"] = SITE_NAME
    
    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": temperature
    }
    
    try:
        async with session.post(
            OPENROUTER_URL,  # Use the full URL with /chat/completions
            headers=headers, 
            json=payload, 
            timeout=aiohttp.ClientTimeout(total=60)
        ) as response:
            # Get response text first for debugging
            response_text = await response.text()
            
            # Check if response is successful
            if response.status != 200:
                print(f"[Error] HTTP {response.status}: {response_text[:200]}")
                return ""
            
            # Try to parse as JSON
            try:
                result = json.loads(response_text)
                return result["choices"][0]["message"]["content"].strip()
            except (json.JSONDecodeError, KeyError) as e:
                print(f"[Error] Failed to parse response: {e}")
                print(f"[Debug] Response text: {response_text[:200]}")
                return ""
                
    except (aiohttp.ClientError, asyncio.TimeoutError) as e:
        print(f"[Error] Request failed: {e}")
        return ""

# === Parsing ===
def parse_choice(response_text, pair):
    """Parse the numerical choice (1 or 2) and map back to original letter."""
    if not response_text:
        return None
    
    clean_response = response_text.strip()
    
    # Direct single digit
    if clean_response in ['1', '2']:
        choice_num = int(clean_response)
        selected_option = pair[choice_num - 1]
        return re.search(r"Option\s+([A-I])", selected_option).group(1)
    
    # Various patterns to extract 1 or 2
    patterns = [
        r'^\s*([12])\s*[.,;:!]?\s*$',  # Just "1" or "2" with optional punctuation
        r'Option\s+([12])\b',  # "Option 1" or "Option 2"
        r'(?:choose|select|prefer|pick)\s+(?:Option\s+)?([12])\b',
        r'(?:My choice is|choice is|choice:)\s+(?:Option\s+)?([12])\b',
        r'\b([12])\b',  # Any "1" or "2" as word boundary
    ]
    
    for pattern in patterns:
        match = re.search(pattern, response_text, re.IGNORECASE | re.MULTILINE)
        if match:
            choice_num = int(match.group(1))
            if choice_num in [1, 2]:
                selected_option = pair[choice_num - 1]
                return re.search(r"Option\s+([A-I])", selected_option).group(1)
    
    return None

# === Infection/NPI helpers ===
def extract_infection_rate(option_text):
    match = re.search(r"(\d+)", option_text)
    return int(match.group(1)) if match else None

def categorize_closure(option_text):
    if "No businesses are closed" in option_text:
        return "No closure"
    elif "Crowded, high-risk businesses" in option_text:
        return "Crowded closures"
    elif "All non-essential businesses" in option_text:
        return "Non-essential closures"
    return "Other"

def npi_level(option_text):
    cat = categorize_closure(option_text)
    return {"No closure": 0, "Crowded closures": 1, "Non-essential closures": 2}.get(cat, -1)

# === CSV Logger ===
def log_response_to_csv(model, pair, selected_choice, response_text, temperature, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    selected_option = canonical_option_map.get(selected_choice)
    if not selected_option:
        print(f"[Warning] Could not find canonical option for {selected_choice}")
        return

    non_selected_option = next((o for o in pair if o != selected_option), None)
    infection_rate_sel = extract_infection_rate(selected_option)
    npi_sel = npi_level(selected_option)
    infection_rate_non = extract_infection_rate(non_selected_option) if non_selected_option else None
    npi_non = npi_level(non_selected_option) if non_selected_option else None
    option_letters = [re.search(r"Option\s+([A-I])", o).group(1) for o in pair]

    entry_id = (
        f"{model.replace('/', '_').replace(':', '_')}_{option_letters[0]}_{option_letters[1]}_"
        f"temp{temperature}_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_{uuid.uuid4().hex[:8]}"
    )

    record = {
        "id": entry_id,
        "timestamp": datetime.utcnow().isoformat(),
        "model": model,
        "temperature": temperature,
        "pair": f"{option_letters[0]}-{option_letters[1]}",
        "option_1": pair[0],
        "option_2": pair[1],
        "selected_choice": selected_choice,
        "response_text": response_text,
        "infection_rate_selected": infection_rate_sel,
        "npi_level_selected": npi_sel,
        "infection_rate_non_selected": infection_rate_non,
        "npi_level_non_selected": npi_non,
    }

    csv_path = os.path.join(output_dir, f"results_{model.replace('/', '_').replace(':', '_')}.csv")
    with csv_lock:
        df = pd.DataFrame([record])
        if not os.path.exists(csv_path):
            df.to_csv(csv_path, index=False)
        else:
            df.to_csv(csv_path, mode="a", index=False, header=False)

# === Async Process Single Pair ===
async def process_pair_async(session, model, pair, temperature, semaphore):
    """Process with semaphore to control concurrency."""
    async with semaphore:  # Limit concurrent requests
        prompt = build_prompt(pair)
        response = await query_openrouter_async(session, model, prompt, temperature)
        
        if not response:
            return False
        
        choice_letter = parse_choice(response, pair)
        if not choice_letter:
            print(f"[Warning] Invalid response for {model} (temp={temperature}): {response[:120]}")
            return False

        log_response_to_csv(model, pair, choice_letter, response, temperature, OUTPUT_DIR)
        return True

# === Async Simulation ===
async def run_simulation_async():
    print(f"Starting simulation with models: {', '.join(MODELS)}")
    print(f"Temperature values: {TEMPERATURE}")
    print(f"Max concurrent requests: {MAX_CONCURRENT}")
    print(f"Base option pairs: {len(base_option_pairs)}")
    print(f"Total option pairs (with reversals): {len(option_pairs)}")

    tasks_data = [(model, pair, temp) for model in MODELS for pair in option_pairs for temp in TEMPERATURE]
    print(f"Total tasks: {len(tasks_data)}")

    # Control concurrency with semaphore
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    
    # Create SSL context using certifi certificates
    ssl_context = ssl.create_default_context(cafile=certifi.where())
    
    successful, failed = 0, 0
    
    # Create connector with SSL context
    connector = aiohttp.TCPConnector(ssl=ssl_context)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [
            process_pair_async(session, model, pair, temp, semaphore)
            for model, pair, temp in tasks_data
        ]
        
        # Process with progress bar using tqdm.asyncio.as_completed
        for coro in tqdm.as_completed(tasks, total=len(tasks), desc="Processing"):
            try:
                result = await coro
                if result:
                    successful += 1
                else:
                    failed += 1
            except Exception as e:
                print(f"[Error] Task failed: {e}")
                failed += 1

    print(f"\nSimulation complete!")
    print(f"Successful: {successful}, Failed: {failed}")
    print(f"Results saved in {OUTPUT_DIR}/")
    for model in MODELS:
        print(f"  - results_{model.replace('/', '_').replace(':', '_')}.csv")


In [10]:

# === Main Entry Point ===
# For Jupyter notebooks: just call await directly
# await run_simulation_async()

# For standalone Python scripts:
if __name__ == "__main__":
    # Check if we're in a Jupyter environment
    try:
        get_ipython()  # This will exist in Jupyter/IPython
        import nest_asyncio
        nest_asyncio.apply()
    except (NameError, ImportError):
        pass  # Not in Jupyter, no need for nest_asyncio
    
    asyncio.run(run_simulation_async())


Starting simulation with models: openai/gpt-5-nano
Temperature values: [0.1, 0.7]
Max concurrent requests: 3
Base option pairs: 36
Total option pairs (with reversals): 72
Total tasks: 144


Processing:   0%|          | 0/144 [00:00<?, ?it/s]/var/folders/6n/bdd22kl9663flsjbqxx1cgdr0000gn/T/ipykernel_3900/1484402887.py:143: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"temp{temperature}_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_{uuid.uuid4().hex[:8]}"
/var/folders/6n/bdd22kl9663flsjbqxx1cgdr0000gn/T/ipykernel_3900/1484402887.py:148: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
Processing: 100%|██████████| 144/144 [13:30<00:00,  5.63s/it]


Simulation complete!
Successful: 144, Failed: 0
Results saved in strategy_one_numerical_results/
  - results_openai_gpt-5-nano.csv
